# EX — Math & Statistics Foundations Real-World Exercises

Linear algebra, probability, hypothesis testing, gradients, and A/B testing — the math
underneath everything else in this course. Requires `numpy`, `scipy`.


## 1. Linear Algebra — Vectors & Matrices as Transformations

In [ ]:
import numpy as np

# A matrix as a transformation: rotate a 2D point by 90 degrees
theta = np.pi/2
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
point = np.array([1, 0])
rotated = R @ point
print("rotated point:", rotated)  # expect roughly [0, 1]


### TODO 1
Compute the eigenvalues/eigenvectors of `R` using `np.linalg.eig`, and separately verify `A @ v = lambda * v` for one eigenpair of a *different*, non-rotation matrix `A = np.array([[4,1],[2,3]])`.

In [ ]:
A = np.array([[4,1],[2,3]])
# TODO
eigenvalues, eigenvectors = None, None
print(eigenvalues, eigenvectors)


<details><summary>Solution</summary>

```python
eigenvalues, eigenvectors = np.linalg.eig(A)
v = eigenvectors[:,0]
lam = eigenvalues[0]
print(np.allclose(A @ v, lam * v))  # should be True
```
</details>

## 2. Probability Distributions

In [ ]:
from scipy import stats

# Normal distribution: heights of adults (cm), mean 170, std 10
heights = stats.norm(loc=170, scale=10)
print("P(height < 160):", heights.cdf(160))
print("P(160 < height < 180):", heights.cdf(180) - heights.cdf(160))

# Poisson: number of support tickets per hour, average rate 3
tickets = stats.poisson(mu=3)
print("P(exactly 5 tickets in an hour):", tickets.pmf(5))


### TODO 2
A call center gets an average of 4 calls per 10-minute window (Poisson). What's the probability of getting **more than 6** calls in a window?

In [ ]:
# TODO
prob_more_than_6 = None
print(prob_more_than_6)


<details><summary>Solution</summary>

```python
calls = stats.poisson(mu=4)
prob_more_than_6 = 1 - calls.cdf(6)
```
</details>

## 3. Bayes' Theorem
Real-world use: spam filtering, medical test interpretation, fraud detection.

In [ ]:
# A medical test is 99% accurate (both sensitivity and specificity).
# The disease affects 1% of the population. If you test positive, what's P(disease | positive)?
p_disease = 0.01
p_positive_given_disease = 0.99
p_positive_given_no_disease = 0.01  # false positive rate

p_no_disease = 1 - p_disease
p_positive = p_positive_given_disease*p_disease + p_positive_given_no_disease*p_no_disease
p_disease_given_positive = (p_positive_given_disease * p_disease) / p_positive
print(f"P(disease | positive test) = {p_disease_given_positive:.3f}")


### TODO 3
Explain in a comment why this probability is much lower than 99%, even though the test is '99% accurate'. Then recompute it for a rarer disease affecting only 0.1% of the population.

In [ ]:
# TODO: your explanation as a comment, then recompute with p_disease = 0.001


<details><summary>Discussion</summary>

Because the disease is rare, the much larger pool of healthy people generates more false positives in absolute terms than the small diseased pool generates true positives — this is the classic 'base rate' intuition behind Bayes' theorem.
</details>

## 4. Hypothesis Testing — Did a Change Actually Help?

In [ ]:
np.random.seed(0)
# Two groups: control (old checkout flow) vs treatment (new checkout flow), conversion times (seconds)
control = np.random.normal(120, 20, 200)
treatment = np.random.normal(114, 20, 200)

t_stat, p_value = stats.ttest_ind(control, treatment)
print(f"t={t_stat:.3f}, p={p_value:.4f}")
print("Significant at 0.05?", p_value < 0.05)


### TODO 4
Compute a 95% confidence interval for the difference in means between `treatment` and `control` (hint: use `stats.sem` for standard error, and a normal approximation: `diff +/- 1.96*SE_diff`).

In [ ]:
# TODO
ci_low, ci_high = None, None
print(ci_low, ci_high)


<details><summary>Solution</summary>

```python
diff = treatment.mean() - control.mean()
se_diff = np.sqrt(control.var(ddof=1)/len(control) + treatment.var(ddof=1)/len(treatment))
ci_low, ci_high = diff - 1.96*se_diff, diff + 1.96*se_diff
```
</details>


## 5. Gradients — Direction of Steepest Increase

In [ ]:
# f(x, y) = x^2 + y^2 -- gradient is [2x, 2y], pointing away from the minimum at (0,0)
def f(x, y): return x**2 + y**2
def grad_f(x, y): return np.array([2*x, 2*y])

point = np.array([3.0, 4.0])
g = grad_f(*point)
print("gradient at (3,4):", g)
print("moving against the gradient (descent) reduces f:", f(*(point - 0.1*g)) < f(*point))


## 6. A/B Test Sample Size — Plan Before You Run
**Pointer:** always compute this *before* launching a test, not after peeking at partial results.

In [ ]:
# Rough sample size estimate for detecting a given effect size at 80% power, alpha=0.05
# Using a simplified formula for two-proportion z-test
def required_sample_size(p1, mde, alpha=0.05, power=0.8):
    from scipy.stats import norm
    p2 = p1 + mde
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    pooled = (p1+p2)/2
    n = ((z_alpha*np.sqrt(2*pooled*(1-pooled)) + z_beta*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2) / (mde**2)
    return int(np.ceil(n))

print(required_sample_size(p1=0.10, mde=0.02))  # baseline 10% conversion, want to detect +2pp


## Key Takeaways
- A matrix is a transformation; eigenvectors are directions that transformation only scales, not rotates.
- Probability distributions model real-world randomness (Poisson for counts, Normal for continuous measurements).
- Bayes' theorem shows why 'high accuracy' tests can still have low positive predictive value for rare events.
- A p-value tells you how surprising your result is under the null hypothesis — not the probability the hypothesis is true.
- Compute required A/B test sample size before launching, not after peeking at early results.
